# noise-batch-from-latent — worked example 1: Flat 2-D latent noise batch for a fully-connected generator

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `noise-batch-from-latent`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

GAN generators require a fresh batch of random latent vectors at every training step. For fully-connected generators the noise has shape `(batch_size, latent_dim)` — a 2-D matrix where each row is an independent sample from the standard normal distribution. Using `torch.randn` gives `N(0, 1)` noise, which is the conventional GAN prior. The batch must be re-sampled each iteration so the generator never sees the same input twice.

## Worked solution

**Step 1 — Choose the right random function.**
We need standard-normal (`N(0,1)`) noise, so we use `t.randn`, not `t.rand` (uniform) or `t.randint`. This matches the GAN convention.

**Step 2 — Specify the shape `(batch_size, latent_dim)`.**
Each row in the resulting matrix is one latent vector. We have `batch_size` rows and `latent_dim` columns. Passing both as positional arguments to `t.randn` gives exactly this shape.

**Step 3 — Send it to the correct device.**
The generator lives on `device` (CPU or CUDA). The noise must be on the same device or the forward pass will raise a device-mismatch error. We pass `device=device` directly to `t.randn` — this is more efficient than creating the tensor on CPU and then calling `.to(device)`.

**Step 4 — Verify and use the result.**
We print the shape and a few statistics to confirm: shape should be `(B, L)`, mean ≈ 0, std ≈ 1.

In [ ]:
import torch as t
import numpy as np

def make_flat_noise(batch_size: int, latent_dim: int, device) -> t.Tensor:
    """Return (batch_size, latent_dim) standard-normal noise on `device`."""
    return t.randn(batch_size, latent_dim, device=device)

# --- exercise it ---
t.manual_seed(42)
device = t.device('cpu')
B, L = 8, 100
noise = make_flat_noise(B, L, device)
print(f'shape : {noise.shape}')       # torch.Size([8, 100])
print(f'mean  : {noise.mean().item():.4f}')   # ≈ 0
print(f'std   : {noise.std().item():.4f}')    # ≈ 1
print(f'device: {noise.device}')       # cpu
assert noise.shape == (B, L)
assert noise.device.type == 'cpu'